In [2]:
import os

# Hide XLA and TensorFlow C++ warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import meshio
from dctkit.mesh import util
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.io as pio
import plotly.graph_objects as go
import dctkit as dt
from functools import partial
from dctkit.math.opt import optctrl as oc
from dctkit.dec import cochain as C
import jax.numpy as jnp

In [3]:
dt.config()

In [4]:
pio.renderers.default = 'notebook_connected' 

In [5]:
# load stanford bunny
filename = "data/stanford-bunny.obj"
mesh = meshio.read(
    filename)
S = util.build_complex_from_mesh(mesh, 3)
S.get_hodge_star()
S.get_complex_boundary_faces_indices()

In [6]:
# 2. Extract boundary nodes
S.get_complex_boundary_faces_indices()
bnd_simplex_indices = S.boundary_simplices[1]
bnodes = np.unique(S.S[1][bnd_simplex_indices])
node_coord = S.node_coords
num_nodes = S.num_nodes

In [33]:

# 1. Prepare the surface data
# S.node_coords contains the (x, y, z) positions
# S.S[2] contains the indices of the triangles
x, y, z = S.node_coords.T
i, j, k = S.S[2].T

# 2. Create the bunny surface
fig = go.Figure(data=[
    go.Mesh3d(
        x=x, y=y, z=z, 
        i=i, j=j, k=k, 
        color='lightpink', 
        opacity=0.50,
        name='Bunny Surface'
    )
])

# 3. Add the boundary nodes as red dots
bnd_coords = S.node_coords[boundary_node_ids]
fig.add_trace(go.Scatter3d(
    x=bnd_coords[:, 0], 
    y=bnd_coords[:, 1], 
    z=bnd_coords[:, 2],
    mode='markers',
    marker=dict(size=3, color='red'),
    name='Boundary Nodes'
))

fig.update_layout(scene=dict(aspectmode='data'))
fig.show()

In [8]:
node_coord = S.node_coords

# 3. Define the Problem Parameters
k = 1.0

# Manufactured exact solution: u = x^2 + y^2 + z^2
u_true = np.array(
    node_coord[:, 0]**2 + node_coord[:, 1]**2 + node_coord[:, 2]**2, 
    dtype=dt.float_dtype
)

# Extract Dirichlet boundary values
b_values = u_true[bnodes]
boundary_values = (np.array(bnodes, dtype=dt.int_dtype), b_values)

# Forcing term: f = -6
f_vec = -6.0 * np.ones((num_nodes,), dtype=dt.float_dtype)

# 4. Define the Energy Formulation for Pygmo
def energy_poisson(x, f, k, boundary_values, gamma, S_complex):
    pos, value = boundary_values
    f_coch = C.Cochain(0, True, S_complex, f)
    u = C.Cochain(0, True, S_complex, x)
    
    # Compute derivative (coboundary)
    du = C.coboundary(u)
    
    # Energy = 0.5 * k * ||du||^2 - (u, f) + penalty
    norm_grad = (k / 2.) * C.inner(du, du)
    bound_term = -C.inner(u, f_coch)
    
    # Penalty method to enforce Dirichlet boundary conditions
    penalty = 0.5 * gamma * jnp.sum((x[pos] - value)**2)
    
    return norm_grad + bound_term + penalty

In [14]:
# Your existing topological check
bnd_simplex_indices = S.boundary_simplices[1]
bnd_simplices_nodes = S.S[1][bnd_simplex_indices]
boundary_node_ids = np.unique(bnd_simplices_nodes)

# SUPPLEMENT: Find nodes at the very bottom of the mesh
z_coords = S.node_coords[:, 2]
z_min = np.min(z_coords)
# Any node within 1% of the bottom is likely part of the base boundary
base_indices = np.where(z_coords < z_min + 0.005)[0]

# Combine them
bnodes = np.unique(np.concatenate([boundary_node_ids, base_indices]))

In [17]:
# 5. Set up the Optimization Problem
gamma = 10000.0
obj = partial(energy_poisson, S_complex=S)
args = {'f': f_vec, 'k': k, 'boundary_values': boundary_values, 'gamma': gamma}

print("Configuring pygmo optimizer...")
prb = oc.OptimizationProblem(dim=num_nodes, state_dim=num_nodes, objfun=obj)
prb.set_obj_args(args)

# 6. Solve
u_0 = 0.01 * np.random.rand(num_nodes).astype(dt.float_dtype)
print("Solving Poisson equation on the Bunny...")
u = prb.solve(u_0, algo="lbfgs", verbose=True, ftol_abs=1e-12, maxeval = 1000).astype(dt.float_dtype)

# Calculate error to verify
error = np.linalg.norm(u - u_true) / np.linalg.norm(u_true)
print(f"Relative error compared to exact solution: {error:.4e}")

Configuring pygmo optimizer...
Solving Poisson equation on the Bunny...
34     0.00152927              0              0
       935     0.00152907              0              0
       936     0.00152892              0              0
       937     0.00152878              0              0
       938     0.00152848              0              0
       939     0.00152805              0              0
       940      0.0015272              0              0
       941       0.001526              0              0
       942     0.00152441              0              0
       943     0.00152453              0              0
       944     0.00152355              0              0
       945     0.00152229              0              0
       946     0.00152136              0              0
       947     0.00152059              0              0
       948     0.00151933              0              0
       949      0.0015178              0              0
       950     0.00151593              0

In [18]:
# 7. Plot the result using Plotly
x, y, z = S.node_coords.T
i, j, k_idx = S.S[2].T

fig = go.Figure(data=[
    go.Mesh3d(
        x=x, y=y, z=z, 
        i=i, j=j, k=k_idx, 
        intensity=u,          # Color the mesh based on the solution 'u'
        colorscale='Viridis', # Color palette for the scalar field
        colorbar_title='u(x,y,z)',
        name='Poisson Solution'
    )
])

fig.update_layout(
    title="Solution to the Poisson Equation on the Stanford Bunny",
    scene=dict(aspectmode='data')
)
fig.show()

In [11]:
prb.last_opt_result

5

In [12]:
u[bnodes]

array([0.00721842, 0.00637423, 0.00597338, 0.00724924, 0.00729495,
       0.00606341, 0.00620248, 0.00677742, 0.00647533, 0.00663381,
       0.00616415, 0.00609078, 0.00603447, 0.00599458, 0.00667695,
       0.00294238, 0.00293589, 0.00136347, 0.00139744, 0.00144034,
       0.00148632, 0.00156099, 0.00158591, 0.0016036 , 0.00161204,
       0.00289786, 0.00298634, 0.00288962, 0.00140814, 0.00171118,
       0.00172364, 0.00174273, 0.00285384, 0.00278583, 0.00285284,
       0.00277504, 0.00131699, 0.00141647, 0.00180606, 0.001818  ,
       0.0027394 , 0.00266278, 0.00273516, 0.00266112, 0.00126375,
       0.00191307, 0.00263292, 0.00255687, 0.00263227, 0.00255881,
       0.00249724, 0.00129471, 0.00197106, 0.00254447, 0.00247166,
       0.00240301, 0.00247311, 0.00240842, 0.00133811, 0.00202924,
       0.00239944, 0.00233562, 0.00239661, 0.00233908, 0.00140945,
       0.00203137, 0.00234109, 0.00227712, 0.00233266, 0.0022754 ,
       0.00149559, 0.00151106, 0.00204803, 0.00229229, 0.00223

In [13]:
u_true[bnodes]

array([0.00721885, 0.00637464, 0.00597364, 0.00725004, 0.00729544,
       0.00606384, 0.00620285, 0.0067781 , 0.00647598, 0.00663419,
       0.00616434, 0.00609096, 0.0060346 , 0.00599488, 0.0066774 ,
       0.00294257, 0.00293614, 0.00136368, 0.00139775, 0.0014407 ,
       0.00148628, 0.00156155, 0.00158534, 0.00160437, 0.00161197,
       0.00289764, 0.00298622, 0.00288944, 0.00140757, 0.00171128,
       0.00172273, 0.00174308, 0.00285299, 0.00278453, 0.00285203,
       0.00277441, 0.00131596, 0.00141544, 0.00180537, 0.00181694,
       0.00273977, 0.00266317, 0.00273464, 0.00266046, 0.00126401,
       0.00191375, 0.00263299, 0.00255765, 0.00263261, 0.00255892,
       0.00249757, 0.00129447, 0.00197102, 0.00254442, 0.00247263,
       0.00240282, 0.00247315, 0.00240854, 0.00133773, 0.00202895,
       0.00239988, 0.00233527, 0.00239655, 0.0023388 , 0.00140957,
       0.00203181, 0.00234093, 0.00227684, 0.00233264, 0.00227536,
       0.00149566, 0.00151076, 0.0020484 , 0.00229266, 0.00223